# Component 3 — Exploratory Data Analysis

This notebook describes the cleaned and harmonized Western Province fisheries modelling candidates before environmental integration. It supports the initial data-analysis section of the viva. Results are exploratory, not causal, and no environmental relationships are tested here.

## Viva workflow position

1. **Audit:** establish available files, schemas, coverage, and quality risks.
2. **Cleaning:** standardize values without altering raw sources.
3. **Harmonization:** create comparable fields and restrict candidates to the study area.
4. **EDA (this notebook):** explain distributions, trends, species, gears, and spatial coverage.
5. **Integration:** attach chlorophyll-a and SST after the fisheries data is understood.

The notebook uses only the model-candidate artifact. The complete all-records artifact remains available for traceability.

In [ ]:
import importlib.util, subprocess, sys
packages={'pandas':'pandas>=2.0,<3','numpy':'numpy>=1.24,<3','pyarrow':'pyarrow>=14,<22','matplotlib':'matplotlib>=3.7,<4'}
missing=[spec for module,spec in packages.items() if importlib.util.find_spec(module) is None]
if missing: subprocess.check_call([sys.executable,'-m','pip','install',*missing])
print('EDA dependencies are available.')

In [ ]:
from __future__ import annotations
import json, os, sys
from pathlib import Path
import matplotlib
if 'IPython' not in sys.modules: matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display=print
pd.set_option('display.max_columns',60)
plt.style.use('seaborn-v0_8-whitegrid')
COLORS={'blue':'#2f6f9f','orange':'#f28e2b','green':'#59a14f','red':'#e15759','purple':'#8f63b8'}

## 1. Locate the processed dataset and output folders

In [ ]:
def find_component_root(start:Path)->Path:
    for anchor in [start.resolve(),*start.resolve().parents]:
        for candidate in [anchor,anchor/'component - 3',anchor/'J26-DS-350'/'component - 3']:
            if (candidate/'Notebooks').is_dir() and (candidate/'src').is_dir(): return candidate.resolve()
    raise FileNotFoundError('Component 3 repository root was not found.')

IN_COLAB='google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT=Path('/content/drive/MyDrive/FishGuard-C3')
    PROCESSED_DIR=ROOT/'data'/'processed'
    HARMONIZATION_AUDIT=ROOT/'outputs'/'audit'/'harmonization'
    EVIDENCE_DIR=ROOT/'outputs'/'audit'/'eda'
    FIGURE_DIR=ROOT/'outputs'/'figures'/'eda'
else:
    COMPONENT_ROOT=find_component_root(Path.cwd())
    PROCESSED_DIR=Path(os.getenv('FISHGUARD_C3_PROCESSED_INPUT',str(COMPONENT_ROOT/'data'/'processed'))).resolve()
    HARMONIZATION_AUDIT=COMPONENT_ROOT/'outputs'/'audit'/'harmonization'
    EVIDENCE_DIR=COMPONENT_ROOT/'outputs'/'audit'/'eda'
    FIGURE_DIR=COMPONENT_ROOT/'outputs'/'figures'/'eda'
for directory in [EVIDENCE_DIR,FIGURE_DIR]: directory.mkdir(parents=True,exist_ok=True)
DATA_PATH=PROCESSED_DIR/'fisheries_harmonized_model_candidates.parquet'
if not DATA_PATH.is_file(): raise FileNotFoundError(f'Missing {DATA_PATH}. Run 02b_semantic_harmonization.ipynb first.')
print('Runtime:','Google Colab' if IN_COLAB else 'Local Windows')
print('Input:',DATA_PATH)

## 2. Load and verify the analytical population

In [ ]:
df=pd.read_parquet(DATA_PATH)
required={'source_year_label','source_row_number','event_date','latitude','longitude','species','catch_weight_kg','gear','vessel_id','model_candidate'}
missing=required-set(df.columns)
if missing: raise KeyError(f'Missing required harmonized fields: {sorted(missing)}')
assert df['model_candidate'].fillna(False).all(),'Processed candidate file contains ineligible rows.'
df['event_date']=pd.to_datetime(df['event_date'],errors='coerce')
df['year']=df['event_date'].dt.year.astype('Int64')
df['month']=df['event_date'].dt.month.astype('Int64')
df['month_name']=pd.Categorical(df['event_date'].dt.strftime('%b'),categories=['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'],ordered=True)
df['positive_weight']=df['catch_weight_kg'].where(df['catch_weight_kg']>0)
df['log1p_weight']=np.log1p(df['catch_weight_kg'].clip(lower=0))
print(f'Rows: {len(df):,}')
print(f'Date coverage: {df.event_date.min().date()} to {df.event_date.max().date()}')
display(df[['source_year_label','event_date','latitude','longitude','species','catch_weight_kg','gear']].head())

## 3. Reproducible descriptive summaries

In [ ]:
overview=pd.DataFrame([
 {'metric':'model_candidate_rows','value':len(df)},
 {'metric':'event_date_start','value':df.event_date.min().isoformat()},
 {'metric':'event_date_end','value':df.event_date.max().isoformat()},
 {'metric':'species_categories','value':df.species.nunique(dropna=True)},
 {'metric':'gear_categories','value':df.gear.nunique(dropna=True)},
 {'metric':'vessels','value':df.vessel_id.nunique(dropna=True)},
 {'metric':'total_catch_weight_kg','value':df.catch_weight_kg.sum()},
 {'metric':'median_catch_weight_kg','value':df.catch_weight_kg.median()},
 {'metric':'zero_weight_rows','value':int(df.catch_weight_kg.eq(0).sum())}
])
annual=(df.groupby('year',observed=True).agg(records=('source_row_number','size'),total_catch_weight_kg=('catch_weight_kg','sum'),median_catch_weight_kg=('catch_weight_kg','median'),species_categories=('species','nunique'),vessels=('vessel_id','nunique')).reset_index())
monthly=(df.groupby(['year','month'],observed=True).agg(records=('source_row_number','size'),total_catch_weight_kg=('catch_weight_kg','sum'),median_catch_weight_kg=('catch_weight_kg','median')).reset_index())
species=(df.groupby('species',dropna=False).agg(records=('source_row_number','size'),total_catch_weight_kg=('catch_weight_kg','sum'),median_catch_weight_kg=('catch_weight_kg','median'),active_years=('year','nunique')).reset_index().sort_values('records',ascending=False))
gear=(df.groupby('gear',dropna=False).agg(records=('source_row_number','size'),total_catch_weight_kg=('catch_weight_kg','sum'),median_catch_weight_kg=('catch_weight_kg','median'),active_years=('year','nunique')).reset_index().sort_values('records',ascending=False))
for table,name in [(overview,'eda_overview.csv'),(annual,'annual_summary.csv'),(monthly,'monthly_summary.csv'),(species,'species_summary.csv'),(gear,'gear_summary.csv')]: table.to_csv(EVIDENCE_DIR/name,index=False)
display(overview); display(annual); display(species.head(15))

## 4. Observation and catch-weight coverage by year

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(15,5))
axes[0].bar(annual.year.astype(str),annual.records,color=COLORS['blue']); axes[0].set_title('Western Province observations by year'); axes[0].set_ylabel('Records')
axes[1].bar(annual.year.astype(str),annual.total_catch_weight_kg,color=COLORS['orange']); axes[1].set_title('Recorded catch weight by year'); axes[1].set_ylabel('Catch weight (kg)')
for ax in axes: ax.grid(axis='y',alpha=.25)
fig.tight_layout(); fig.savefig(FIGURE_DIR/'annual_coverage.png',dpi=200,bbox_inches='tight'); plt.show()

## 5. Catch-weight distribution

In [ ]:
positive=df.loc[df.positive_weight.notna(),'positive_weight']
upper=positive.quantile(.99) if len(positive) else 1
fig,axes=plt.subplots(1,2,figsize=(15,5))
axes[0].hist(positive.clip(upper=upper),bins=60,color=COLORS['blue']); axes[0].set_title('Positive catch weight (capped at 99th percentile for display)'); axes[0].set_xlabel('Catch weight (kg)'); axes[0].set_ylabel('Records')
axes[1].hist(np.log10(positive),bins=60,color=COLORS['purple']); axes[1].set_title('Positive catch weight on log10 scale'); axes[1].set_xlabel('log10(catch weight in kg)')
fig.tight_layout(); fig.savefig(FIGURE_DIR/'catch_weight_distribution.png',dpi=200,bbox_inches='tight'); plt.show()
print(f'99th percentile used only for display: {upper:,.2f} kg. Source values were not modified.')

## 6. Monthly pattern

In [ ]:
month_summary=(df.groupby('month_name',observed=False).agg(records=('source_row_number','size'),median_catch_weight_kg=('catch_weight_kg','median')).reset_index())
month_summary.to_csv(EVIDENCE_DIR/'calendar_month_summary.csv',index=False)
fig,axes=plt.subplots(1,2,figsize=(15,5))
axes[0].bar(month_summary.month_name.astype(str),month_summary.records,color=COLORS['green']); axes[0].set_title('Observations by calendar month'); axes[0].set_ylabel('Records')
axes[1].plot(month_summary.month_name.astype(str),month_summary.median_catch_weight_kg,marker='o',color=COLORS['orange']); axes[1].set_title('Median recorded catch weight by month'); axes[1].set_ylabel('Median catch weight (kg)')
for ax in axes: ax.grid(axis='y',alpha=.25)
fig.tight_layout(); fig.savefig(FIGURE_DIR/'monthly_patterns.png',dpi=200,bbox_inches='tight'); plt.show()

## 7. Species and gear composition

In [ ]:
top_species=species.head(15).sort_values('records')
top_gear=gear.head(12).sort_values('records')
fig,axes=plt.subplots(1,2,figsize=(16,7))
axes[0].barh(top_species.species.astype(str),top_species.records,color=COLORS['blue']); axes[0].set_title('Most frequently recorded species categories'); axes[0].set_xlabel('Records')
axes[1].barh(top_gear.gear.astype(str),top_gear.records,color=COLORS['orange']); axes[1].set_title('Most frequently recorded gear categories'); axes[1].set_xlabel('Records')
fig.tight_layout(); fig.savefig(FIGURE_DIR/'species_and_gear_composition.png',dpi=200,bbox_inches='tight'); plt.show()

## 8. Spatial observation and catch-weight patterns

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(15,6),sharex=True,sharey=True)
h1=axes[0].hexbin(df.longitude,df.latitude,gridsize=35,mincnt=1,cmap='Blues'); axes[0].set_title('Observation density'); fig.colorbar(h1,ax=axes[0],label='Records per hexagon')
h2=axes[1].hexbin(df.longitude,df.latitude,C=df.catch_weight_kg,reduce_C_function=np.median,gridsize=35,mincnt=1,cmap='YlOrRd'); axes[1].set_title('Median catch weight by location'); fig.colorbar(h2,ax=axes[1],label='Median catch weight (kg)')
for ax in axes: ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude'); ax.grid(alpha=.2)
fig.tight_layout(); fig.savefig(FIGURE_DIR/'spatial_patterns.png',dpi=200,bbox_inches='tight'); plt.show()

## 9. Harmonization retention context

In [ ]:
retention_path=HARMONIZATION_AUDIT/'harmonization_summary.csv'
if retention_path.is_file():
    retention=pd.read_csv(retention_path)
    main=retention.loc[retention.source_role.eq('candidate_observations')].copy()
    main['study_area_percent']=100*main.study_area_rows/main.rows
    main['model_candidate_percent']=100*main.model_candidates/main.rows
    main.to_csv(EVIDENCE_DIR/'retention_context.csv',index=False)
    display(main[['source_year_label','rows','study_area_rows','model_candidates','study_area_percent','model_candidate_percent']])
else:
    print('Harmonization summary was not available; EDA results remain valid but retention context was skipped.')

## 10. Save manifest and viva findings

In [ ]:
top_species_name=str(species.iloc[0].species) if len(species) else None
top_gear_name=str(gear.iloc[0].gear) if len(gear) else None
findings=pd.DataFrame([
 {'finding':'Analytical population','result':f'{len(df):,} harmonized Western Province model candidates'},
 {'finding':'Temporal coverage','result':f'{df.event_date.min().date()} to {df.event_date.max().date()}'},
 {'finding':'Largest annual sample','result':f'{int(annual.loc[annual.records.idxmax(),"year"])} ({int(annual.records.max()):,} records)'},
 {'finding':'Most recorded species category','result':top_species_name},
 {'finding':'Most recorded gear category','result':top_gear_name},
 {'finding':'Median catch weight','result':f'{df.catch_weight_kg.median():,.2f} kg'},
 {'finding':'Interpretation limit','result':'Exploratory associations only; environmental variables are not yet integrated'}
])
findings.to_csv(EVIDENCE_DIR/'viva_findings.csv',index=False)
manifest={'created_utc':pd.Timestamp.now(tz='UTC').isoformat(),'runtime':'Google Colab' if IN_COLAB else 'Local Windows','input':DATA_PATH.name,'rows':len(df),'date_start':df.event_date.min().isoformat(),'date_end':df.event_date.max().isoformat(),'outputs':sorted([p.name for p in EVIDENCE_DIR.glob('*') if p.is_file()]),'figures':sorted([p.name for p in FIGURE_DIR.glob('*.png')])}
with (EVIDENCE_DIR/'eda_manifest.json').open('w',encoding='utf-8') as f: json.dump(manifest,f,indent=2)
display(findings)
print('EDA evidence:',EVIDENCE_DIR)
print('EDA figures:',FIGURE_DIR)

## Interpretation rules for the viva

- These results describe the available, filtered records; they do not prove changes in the underlying fish population.
- Annual totals are affected by different sample sizes and recording coverage. Compare medians and record counts alongside totals.
- The catch-weight distribution is expected to be skewed, so logarithmic and robust summaries are shown.
- Vessel identifiers are aggregated and never displayed.
- Spatial patterns describe the provisional Western Province coastal window and should not be generalized to all Sri Lankan waters.
- Environmental relationships must be discussed only after `03_data_integration.ipynb` is completed.